# 🔍 Notebook 7: Hybrid Retrieval System (Books)
**Goal:** Combine ChromaDB semantic search over textbook chunks + NetworkX Knowledge Graph node lookup into a unified retrieve function.

**Input:** A query string
**Output:** Formatted context string + `retriever.py` module for pipeline integration

In [ ]:
import os
import pickle
import warnings
import chromadb
from sentence_transformers import SentenceTransformer
import networkx as nx
import torch

warnings.filterwarnings("ignore")
print("✅ Required libraries loaded")

## 1. Load Knowledge Graph (From Books)

In [ ]:
KG_PATH = "data/kg_medical_books.pkl"
if os.path.exists(KG_PATH):
    with open(KG_PATH, "rb") as f:
        G = pickle.load(f)
    print(f"✅ Medical Book KG loaded: {len(G.nodes)} nodes")
else:
    G = nx.DiGraph()  # Empty fallback
    print("⚠️ KG not found. Run Notebook 03 first.")

# Pre-calculate lower-case node names for faster matching
node_names = {str(n).lower(): n for n in G.nodes()}

## 2. Load Vector DB (From Books)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer("intfloat/multilingual-e5-large", device=device)

try:
    client = chromadb.PersistentClient(path="data/chroma_db_books")
    collection = client.get_collection(name="medical_books")
    print(f"✅ ChromaDB loaded: {collection.count()} book chunks.")
except Exception as e:
    collection = None
    print(f"⚠️ Warning: ChromaDB not found or empty (Run Notebook 06 first). Error: {e}")

## 3. Hybrid Retrieval Concept (Vector + Graph)

In [ ]:
def extract_kg_subgraph(query: str):
    """
    Matches query keywords against KG nodes. Returns a summary of 1-hop neighbors.
    """
    query_lower = query.lower()
    matched_nodes = []

    # Substring matching to find relevant nodes
    for node_lower, actual_node in node_names.items():
        if node_lower in query_lower:
            matched_nodes.append(actual_node)

    # Extract knowledge facts for matched nodes
    kg_facts = set()
    diseases_found = []
    treatments_found = []

    for node in matched_nodes:
        node_type = G.nodes[node].get("type", "entity")

        # Check outgoing edges (what does this cause/treat?)
        for neighbor in G.successors(node):
            relation = G.edges[node, neighbor].get("relation", "related_to")
            kg_facts.add(f"{node} {relation} {neighbor}")
            if G.nodes[neighbor].get("type") == "disease":
                diseases_found.append(neighbor)
            if G.nodes[neighbor].get("type") == "treatment":
                treatments_found.append(neighbor)

        # Check incoming edges (what causes/treats this?)
        for neighbor in G.predecessors(node):
            relation = G.edges[neighbor, node].get("relation", "related_to")
            kg_facts.add(f"{neighbor} {relation} {node}")
            if G.nodes[neighbor].get("type") == "disease":
                diseases_found.append(neighbor)

    return {
        "matched_entities": list(set(matched_nodes)),
        "facts": list(kg_facts),
        "possible_diseases": list(set(diseases_found)),
        "suggested_treatments": list(set(treatments_found)),
    }


def vector_search(query: str, top_k: int = 5):
    """
    Retrieves top-k text chunks from medical books.
    """
    if collection is None:
        return []

    query_vec = embed_model.encode(f"query: {query}", convert_to_numpy=True).tolist()
    results = collection.query(query_embeddings=[query_vec], n_results=top_k)

    docs = []
    for doc, dist, meta in zip(
        results["documents"][0], results["distances"][0], results["metadatas"][0]
    ):
        docs.append(
            {
                "text": doc,
                "score": float(dist),
                "source": meta.get("source", "Unknown Book"),
            }
        )
    return docs

## 4. Run Hybrid Pipeline

In [ ]:
def hybrid_retrieve(query: str, top_k: int = 5):
    kg_data = extract_kg_subgraph(query)
    v_data = vector_search(query, top_k=top_k)

    # Combine into unified string prompt
    context_blocks = []

    if kg_data["facts"]:
        block = "[KNOWLEDGE GRAPH MEDICAL FACTS]\n"
        block += "\n".join([f"- {fact}" for fact in kg_data["facts"]])
        context_blocks.append(block)

    if v_data:
        block = "[MEDICAL BOOK EXTRACTS]\n"
        for i, d in enumerate(v_data):
            block += f"Extract {i + 1} (Source: {d['source']}): {d['text']}\n"
        context_blocks.append(block)

    combined = (
        "\n\n".join(context_blocks) if context_blocks else "No specific context found."
    )

    return {
        "query": query,
        "combined_context": combined,
        "kg_results": kg_data,
        "vector_results": v_data,
    }


# Test it
test_response = hybrid_retrieve("ડાયાબિટીઝ ના લક્ષણો")
print("✅ Hybrid Pipeline Combined Context Result:")
print("=" * 50)
print(test_response["combined_context"])
print("=" * 50)

## 5. Export as `retriever.py`

In [ ]:
module_code = '''
"""retriever.py — Hybrid Knowledge Graph + Vector DB Retriever for Medical Books"""
import os, pickle
import chromadb
from sentence_transformers import SentenceTransformer
import networkx as nx
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
_embed_model = SentenceTransformer("intfloat/multilingual-e5-large", device=device)

# Load KG
if os.path.exists("data/kg_medical_books.pkl"):
    with open("data/kg_medical_books.pkl", "rb") as f: _G = pickle.load(f)
else:
    _G = nx.DiGraph()
_node_names = {str(n).lower(): n for n in _G.nodes()}

# Load Vector DB
try:
    _client = chromadb.PersistentClient(path="data/chroma_db_books")
    _collection = _client.get_collection(name="medical_books")
except:
    _collection = None

def extract_kg_subgraph(query: str):
    q_low = query.lower()
    matched = [actual for q_node, actual in _node_names.items() if q_node in q_low]
    
    facts, dis, trt = set(), [], []
    for n in matched:
        for neighbor in _G.successors(n):
            rel = _G.edges[n, neighbor].get("relation", "related_to")
            facts.add(f"{n} {rel} {neighbor}")
            if _G.nodes[neighbor].get("type") == "disease": dis.append(neighbor)
            if _G.nodes[neighbor].get("type") == "treatment": trt.append(neighbor)
        for neighbor in _G.predecessors(n):
            rel = _G.edges[neighbor, n].get("relation", "related_to")
            facts.add(f"{neighbor} {rel} {n}")
            if _G.nodes[neighbor].get("type") == "disease": dis.append(neighbor)
            
    return {"matched_entities": list(set(matched)), "facts": list(facts),
            "possible_diseases": list(set(dis)), "suggested_treatments": list(set(trt))}

def vector_search(query: str, top_k: int = 5):
    if not _collection: return []
    q_vec = _embed_model.encode(f"query: {query}", convert_to_numpy=True).tolist()
    res = _collection.query(query_embeddings=[q_vec], n_results=top_k)
    
    docs = []
    for doc, dist, meta in zip(res["documents"][0], res["distances"][0], res["metadatas"][0]):
        docs.append({"text": doc, "score": float(dist), "source": meta.get("source", "Book")})
    return docs

def hybrid_retrieve(query: str, top_k: int = 5):
    kg = extract_kg_subgraph(query)
    vdb = vector_search(query, top_k)
    
    blocks = []
    if kg["facts"]:
        blocks.append("[KNOWLEDGE GRAPH MEDICAL FACTS]\n" + "\n".join([f"- {f}" for f in kg["facts"]]))
    if vdb:
        block = "[MEDICAL BOOK EXTRACTS]\n"
        for i, d in enumerate(vdb): block += f"Extract {i+1} (Source: {d['source']}): {d['text']}\n"
        blocks.append(block)
        
    return {
        "query": query, 
        "combined_context": "\n\n".join(blocks) if blocks else "",
        "kg_results": kg, "vector_results": vdb
    }
'''

with open("retriever.py", "w", encoding="utf-8") as f:
    f.write(module_code.strip())

print("✅ retriever.py module saved successfully!")
print("\n📝 NEXT STEP: Run Notebook 08 to combine Retriever with Qwen 2.5 LLM.")